# 20대 건강 데이터 시각화 (코랩 실행용)

**준비**: 왼쪽 📁 파일 탭에 `health20s.zip` 과 데이터 파일 8개(CSV/XLSX)를 업로드한 뒤 위에서부터 차례로 실행하세요.
(데이터 파일은 `/content` 바로 아래에 두면 자동으로 찾아요)

In [ ]:
# 1) 압축 풀기 + 스트림릿 설치
!unzip -o -q /content/health20s.zip -d /content
!pip -q install streamlit
%cd /content/health20s

In [ ]:
# 2) 한글 폰트 설정 - 런타임 재시작 필요 없음
from korean_font import setup_korean_font
setup_korean_font()

In [ ]:
# 3) 데이터 확인 (2018년 추정값 = 추정 열 True)
import data
m = data.mets()
m[m['연도'].between(2017, 2019) & (m['성별'] == '전체')]

## PNG 그래프

In [ ]:
# 4) 그래프 8개를 figures/ 에 PNG로 저장하고 화면에 표시
import make_png
from IPython.display import Image, display
for p in make_png.main('figures'):
    display(Image(p))

In [ ]:
# (선택) 그래프 하나만 바로 보기
import charts
charts.fig_mets();

In [ ]:
# (선택) PNG 전부 zip으로 내려받기
!zip -q -r figures.zip figures
from google.colab import files
files.download('figures.zip')

## API 서버

In [ ]:
# 5) API 서버를 백그라운드로 켜기
!nohup python api_server.py 5000 > api.log 2>&1 &
import time, requests; time.sleep(5)
print(requests.get('http://localhost:5000/api/summary').json())

In [ ]:
# 6) 브라우저에서 API 열어 보기 (아래 링크 클릭 -> 뒤에 /api/indicators 등을 붙이면 됨)
from google.colab.output import eval_js
print(eval_js('google.colab.kernel.proxyPort(5000)'))

In [ ]:
# 7) API로 받은 데이터로 그래프 그리기 예시
import pandas as pd
r = requests.get('http://localhost:5000/api/indicators/f32').json()
df = pd.DataFrame(r['data'])
charts.fig_f32(df);

## 스트림릿 대시보드

아래 셀을 실행하면 `your url is: https://....loca.lt` 링크가 나와요.
링크에 들어가서 **Tunnel Password** 칸에 바로 위에 출력된 숫자(IP 주소)를 넣으면 대시보드가 열립니다.
사이드바에서 '데이터 출처 → API 서버'를 고르면 5번에서 켠 API를 통해 데이터를 받아 그려요.

In [ ]:
# 8) 스트림릿 실행 + 외부 접속 링크 만들기 (셀을 멈추면 링크도 꺼짐)
!nohup streamlit run streamlit_app.py --server.port 8501 --server.headless true > st.log 2>&1 &
!sleep 5 && curl -s https://loca.lt/mytunnelpassword && echo
!npx -y localtunnel --port 8501